# Aggregate new-data site alarm-correlation results

Exact notebook form of `aggregate_alarm_correlation_results.py`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root")


WORKSPACE = find_project_root()
RESULTS_DIR = WORKSPACE / "results"


def load_site_tables(filename):
    frames = []
    for site_dir in sorted(RESULTS_DIR.glob("site_*")):
        path = site_dir / filename
        if not path.exists():
            continue
        try:
            frame = pd.read_csv(path)
        except pd.errors.EmptyDataError:
            continue
        if "site_id" not in frame.columns:
            frame.insert(0, "site_id", int(site_dir.name.split("_")[1]))
        frames.append(frame)
    return pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()


summaries = load_site_tables("site_analysis_summary.csv")
associations = load_site_tables("sampled_parameter_alarm_associations.csv")
onset_findings = load_site_tables("parameter_alarm_onset_findings.csv")
alarm_type_findings = load_site_tables("alarm_type_parameter_findings.csv")

summaries.to_csv(RESULTS_DIR / "all_sites_analysis_summary.csv", index=False)
associations.to_csv(RESULTS_DIR / "all_sites_sampled_parameter_associations.csv", index=False)
onset_findings.to_csv(RESULTS_DIR / "all_sites_alarm_onset_findings.csv", index=False)
alarm_type_findings.to_csv(
    RESULTS_DIR / "all_sites_alarm_type_parameter_findings.csv", index=False
)

if not onset_findings.empty:
    candidate_thresholds = onset_findings[
        (onset_findings["mapped_alarm_values"] >= 10)
        & (onset_findings["evidence_score"] > 0)
    ].copy()
    candidate_thresholds.sort_values(
        ["site_id", "evidence_score"], ascending=[True, False], inplace=True
    )
    top_candidates = candidate_thresholds.groupby("site_id", as_index=False).head(3)
else:
    candidate_thresholds = pd.DataFrame()
    top_candidates = pd.DataFrame()

candidate_thresholds.to_csv(
    RESULTS_DIR / "all_sites_candidate_thresholds.csv", index=False
)
top_candidates.to_csv(
    RESULTS_DIR / "all_sites_top_three_candidate_thresholds.csv", index=False
)


if not associations.empty:
    heatmap = associations.pivot_table(
        index="site_id",
        columns="parameter",
        values="point_biserial_r",
        aggfunc="first",
    ).sort_index()
    fig, axis = plt.subplots(
        figsize=(max(10, len(heatmap.columns) * 1.45), max(6, len(heatmap.index) * 0.55))
    )
    image = axis.imshow(heatmap.fillna(0), aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)
    axis.set_xticks(np.arange(len(heatmap.columns)))
    axis.set_xticklabels(heatmap.columns, rotation=45, ha="right")
    axis.set_yticks(np.arange(len(heatmap.index)))
    axis.set_yticklabels(heatmap.index)
    axis.set_title("Sampled parameter correlation with alarm-active status")
    axis.set_xlabel("Parameter")
    axis.set_ylabel("Site ID")
    fig.colorbar(image, ax=axis, label="Point-biserial correlation")
    fig.tight_layout()
    fig.savefig(
        RESULTS_DIR / "all_sites_sampled_correlation_heatmap.png",
        dpi=160,
        bbox_inches="tight",
    )
    plt.close(fig)


if not top_candidates.empty:
    best_per_site = (
        top_candidates.sort_values("evidence_score", ascending=False)
        .groupby("site_id", as_index=False)
        .head(1)
        .sort_values("evidence_score")
    )
    labels = [
        f"{int(row.site_id)}: {row.parameter}"
        for row in best_per_site.itertuples(index=False)
    ]
    fig, axis = plt.subplots(figsize=(13, max(6, len(best_per_site) * 0.55)))
    colors = [
        "#dc2626" if "high values" in str(direction) else "#2563eb"
        for direction in best_per_site["candidate_direction"]
    ]
    axis.barh(labels, best_per_site["evidence_score"], color=colors)
    axis.set_xlabel("Exploratory onset evidence score")
    axis.set_title("Strongest parameter pattern at exact alarm starts by site")
    axis.grid(True, axis="x", alpha=0.2)
    fig.tight_layout()
    fig.savefig(
        RESULTS_DIR / "all_sites_top_alarm_onset_candidates.png",
        dpi=160,
        bbox_inches="tight",
    )
    plt.close(fig)

print("Sites summarized:", len(summaries))
print("Association rows:", len(associations))
print("Onset finding rows:", len(onset_findings))
print("Candidate threshold rows:", len(candidate_thresholds))
